# Retrieval Test

검색 함수 자체는 `src/retriever.py`에 정의되어 있고, 여기서는 호출만 한다.
동일 함수를 노트북/스크립트/파이프라인 모두 공유 → 중복 없음.

In [ ]:
# 프로젝트 루트를 sys.path 에 추가 (notebooks/ 하위에서 src import)
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
from src.retriever import retrieve, hybrid_rrf_search
from src.es_client import get_es

es = get_es()
print('ES:', es.info()['version']['number'])

In [ ]:
query = "What's the difference between Market Cap and NAV?"

for mode in ['dense', 'bm25', 'elser', 'hybrid']:
    print('\n' + '='*60)
    print(f'[{mode}]')
    try:
        for i, d in enumerate(retrieve(query, mode=mode, top_k=5), 1):
            print(f"[{i}] score={d['score']:.4f}  id={d['doc_id']}")
            print(f"    {d['text'][:200]}")
    except Exception as e:
        print('ERROR:', e)

In [ ]:
# 하이브리드 가중치 튜닝 실험
for dw, bw in [(0.7, 0.3), (0.5, 0.5), (0.3, 0.7)]:
    print(f'\n--- dense={dw}, bm25={bw} ---')
    res = hybrid_rrf_search(query, top_k=10, final_k=5, dense_weight=dw, bm25_weight=bw)
    for i, d in enumerate(res, 1):
        print(f"[{i}] rrf={d['rrf_score']:.5f}  dense_rank={d['dense_rank']}  bm25_rank={d['bm25_rank']}")